<a href="https://colab.research.google.com/github/Rimsha2803/data-analytics-portfolio/blob/main/hypothesis-testing/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
uploaded = files.upload()
df = pd.read_csv('online_retail_cleaned.XSLX.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.shape

Saving online_retail_cleaned.XSLX.csv to online_retail_cleaned.XSLX.csv


/tmp/ipykernel_4212/3035751076.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('online_retail_cleaned.XSLX.csv')


(524878, 14)

In [ ]:
df_rfm_base = df[df['is_guest'] == False].copy()
df_rfm_base['CustomerID'] = df_rfm_base['CustomerID'].astype(int)
df_rfm_base.shape

(392692, 14)

In [ ]:
snapshot_date = df_rfm_base['InvoiceDate'].max() + pd.Timedelta(days=1)
snapshot_date

Timestamp('2011-12-10 12:50:00')

In [ ]:
df_rfm_base['Revenue'] = df_rfm_base['Quantity'] * df_rfm_base['UnitPrice']

rfm = df_rfm_base.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

def rfm_segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'
    elif row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'Loyal Customers'
    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return 'New Customers'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)
rfm['Segment'].value_counts()

,count
Segment,
Lost,1074
Loyal Customers,999
Champions,943
At Risk,661
Needs Attention,351
New Customers,310


In [ ]:
customer_country = df_rfm_base.groupby('CustomerID')['Country'].first().reset_index()
rfm = rfm.merge(customer_country, on='CustomerID', how='left')

rfm.to_csv('rfm_table.csv', index=False)
files.download('rfm_table.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(rfm.shape)
print(rfm['Segment'].value_counts())

(4338, 9)
Segment
Lost               1074
Loyal Customers     999
Champions           943
At Risk             661
Needs Attention     351
New Customers       310
Name: count, dtype: int64


In [ ]:
# Reload transaction-level data (not the RFM table) for cohort analysis
df2 = pd.read_csv('online_retail_cleaned.XSLX.csv')
df2['InvoiceDate'] = pd.to_datetime(df2['InvoiceDate'])
df2 = df2[df2['is_guest'] == False].copy()
df2['CustomerID'] = df2['CustomerID'].astype(int)

# Each customer's first purchase month = their "cohort"
df2['InvoiceMonth'] = df2['InvoiceDate'].dt.to_period('M')
df2['CohortMonth'] = df2.groupby('CustomerID')['InvoiceMonth'].transform('min')

# How many months after their first purchase is this transaction?
df2['CohortIndex'] = (
    (df2['InvoiceMonth'].dt.year - df2['CohortMonth'].dt.year) * 12
    + (df2['InvoiceMonth'].dt.month - df2['CohortMonth'].dt.month)
)

# Count active customers per cohort, per month-since-first-purchase
cohort_data = df2.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_data.rename(columns={'CustomerID': 'ActiveCustomers'}, inplace=True)
cohort_data['CohortMonth'] = cohort_data['CohortMonth'].astype(str)

cohort_data.to_csv('cohort_data.csv', index=False)
files.download('cohort_data.csv')

cohort_data.head(10)

/tmp/ipykernel_4212/186817425.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('online_retail_cleaned.XSLX.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,CohortMonth,CohortIndex,ActiveCustomers
0,2010-12,0,885
1,2010-12,1,324
2,2010-12,2,286
3,2010-12,3,340
4,2010-12,4,321
5,2010-12,5,352
6,2010-12,6,321
7,2010-12,7,309
8,2010-12,8,313
9,2010-12,9,350


In [ ]:
print(rfm.shape)
print(df2.shape)

(4338, 9)
(392692, 16)


In [ ]:
# Attach each customer's Segment to their transaction-level cohort data
df2_seg = df2.merge(rfm[['CustomerID', 'Segment']], on='CustomerID', how='left')

# Rebuild cohort counts, now broken out by Segment too
cohort_data_seg = (
    df2_seg.groupby(['Segment', 'CohortMonth', 'CohortIndex'])['CustomerID']
    .nunique()
    .reset_index()
)
cohort_data_seg.rename(columns={'CustomerID': 'ActiveCustomers'}, inplace=True)
cohort_data_seg['CohortMonth'] = cohort_data_seg['CohortMonth'].astype(str)

cohort_data_seg.to_csv('cohort_data_by_segment.csv', index=False)
files.download('cohort_data_by_segment.csv')

cohort_data_seg.head(10)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Segment,CohortMonth,CohortIndex,ActiveCustomers
0,At Risk,2010-12,0,167
1,At Risk,2010-12,1,49
2,At Risk,2010-12,2,44
3,At Risk,2010-12,3,48
4,At Risk,2010-12,4,43
5,At Risk,2010-12,5,46
6,At Risk,2010-12,6,45
7,At Risk,2010-12,7,29
8,At Risk,2010-12,8,41
9,At Risk,2010-12,9,31


In [ ]:
print(cohort_data_seg.shape)
print(cohort_data_seg['Segment'].isnull().sum())
cohort_data_seg.head(10)

(352, 4)
0


,Segment,CohortMonth,CohortIndex,ActiveCustomers
0,At Risk,2010-12,0,167
1,At Risk,2010-12,1,49
2,At Risk,2010-12,2,44
3,At Risk,2010-12,3,48
4,At Risk,2010-12,4,43
5,At Risk,2010-12,5,46
6,At Risk,2010-12,6,45
7,At Risk,2010-12,7,29
8,At Risk,2010-12,8,41
9,At Risk,2010-12,9,31


In [ ]:
print(rfm.shape)
print(cohort_data_seg.shape)

(4338, 9)
(352, 4)


### Task 3: Customer Segmentation & Behavioral Trend Dashboard

**RFM Segmentation**

Recency, Frequency, and Monetary metrics were computed per customer
from the cleaned Task 2 dataset, excluding guest checkouts (customers
without an identified CustomerID), leaving 4,338 identified customers.
Each metric was scored into quintiles (1-5) using rank-based binning
to avoid errors from tied values, then combined into six segments:
Lost (1,074), Loyal Customers (999), Champions (943), At Risk (661),
Needs Attention (351), and New Customers (310).

Lost customers make up the largest single segment (~25% of the
customer base), just ahead of Loyal Customers and Champions — a
meaningful churn signal worth flagging as a business finding.

**Recency vs Monetary**

An unaggregated scatter plot of all 4,338 customers, colored by
segment, confirmed Champions cluster in the low-recency/high-monetary
region as expected. Monetary value remains heavily right-skewed at the
customer level, consistent with the UnitPrice/Quantity skew already
found in Task 2 — a small number of high-spending customers pull the
distribution's tail, while most customers cluster near zero.

**Geospatial Distribution**

A country-level map (built from Tableau's automatic geocoding of the
Country field) confirmed the United Kingdom dominates the customer
base by a wide margin, consistent with this being a UK-based retailer.

**Cohort Retention**

Customers were grouped by the month of their first purchase (cohort),
then tracked by how many remained active in each following month. The
resulting heatmap showed a consistent, steep first-month drop-off
across nearly every cohort (e.g., the December 2010 cohort fell from
885 active customers to 324 within one month, a ~63% drop), with
retention leveling off more gradually afterward. The December 2010
cohort itself is somewhat inflated, since it is the first month the
dataset records — it includes pre-existing customers who simply
happened to purchase that month, not genuinely new customers, which
is noted here as a limitation rather than a true "best" cohort.

**Interactive Dashboard**

All four views (segment overview, recency-vs-monetary scatter,
customer map, and cohort retention) were combined into a single
Tableau Public dashboard, built entirely through browser-based Web
Authoring due to hardware constraints. Cohort retention was
additionally broken out by customer segment and linked via a dashboard
filter action, so selecting a segment in the overview chart
cross-filters all four views simultaneously — fulfilling the brief's
requirement for interactive, linked charts alongside the geospatial
map and cohort trend.